In [1]:
#!pip install uszipcode
#!pip install 'sqlalchemy_mate < 2.0.0.1'

In [2]:
import pandas as pd
from uszipcode import SearchEngine

/opt/anaconda3/lib/python3.13/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [3]:
cdc_24 = pd.read_csv('../data/cleaned/cdc_cleaned_2024.csv')
cdc_24.head(2)

,county,fips,Sex,Sex Code,year,Year Code,Cause of death,Cause of death Code,Place of Death,Place of Death Code,deaths
0,"Jefferson County, AL",1073.0,Female,F,2024.0,2024.0,Accidental poisoning by and exposure to narcot...,X42,Decedent's home,4.0,15.0
1,"Jefferson County, AL",1073.0,Female,F,2024.0,2024.0,Accidental poisoning by and exposure to narcot...,X42,Other,7.0,12.0


In [4]:
facilities = pd.read_csv('../data/cleaned/samhsa_facilities_zip.csv')
facilities.head(2)

,state,zip,facility_count
0,AK,99501,2
1,AK,99503,6


In [5]:
census_24 = pd.read_csv('../data/cleaned/census_acs_2024.csv')
census_24.head(2)

,county_name,median_income,population,state,county,poverty_rate,unemployment_rate,fips
0,"Autauga County, Alabama",72481,59947,1,1,11.3,2.4,1001
1,"Baldwin County, Alabama",78775,246989,1,3,10.1,3.0,1003


## To begin this notebook I need to make sure that my zip codes are able to be merged on for all of my datasets.

In [6]:
search = SearchEngine()

facilities["county"] = (
    facilities["zip"]
    .apply(
        lambda z:
        search.by_zipcode(z).county
        if search.by_zipcode(z)
        else None
    )
)

facilities.head()

,state,zip,facility_count,county
0,AK,99501,2,Anchorage Municipality
1,AK,99503,6,Anchorage Municipality
2,AK,99507,1,Anchorage Municipality
3,AK,99508,10,Anchorage Municipality
4,AK,99518,2,Anchorage Municipality


## Let's attach County FIPS

In [7]:
county_lookup = census_24[
    ["county","state","fips"]
].copy()

# Convert to string first, then apply .str.upper() to handle non-string values
county_lookup["county"] = (
    county_lookup["county"]
    .astype(str)  # Convert all values to string first
    .str.upper()
)

# Convert to string first, then apply .str.upper() to handle non-string values
facilities["county"] = (
    facilities["county"]
    .astype(str)  # Convert all values to string first
    .str.upper()
)

In [8]:
cdc_24["fips"] = (
    cdc_24["fips"]
    .astype(str)
    .str.zfill(5)
)

census_24["fips"] = (
    census_24["fips"]
    .astype(str)
    .str.zfill(5)
)

facilities["fips"] = (
    facilities["zip"]
    .astype(str)
    .str.zfill(5)
)

In [9]:
facilities.head(2)

,state,zip,facility_count,county,fips
0,AK,99501,2,ANCHORAGE MUNICIPALITY,99501
1,AK,99503,6,ANCHORAGE MUNICIPALITY,99503


## Merging 

In [10]:
county_lookup.head(2)

,county,state,fips
0,1,1,1001
1,3,1,1003


In [11]:
facilities.head(2)

,state,zip,facility_count,county,fips
0,AK,99501,2,ANCHORAGE MUNICIPALITY,99501
1,AK,99503,6,ANCHORAGE MUNICIPALITY,99503


In [12]:
facilities = (
    facilities
    .merge(
        county_lookup,
        on="county",
        how="left"
    )
)

facilities.isna().sum()

state_x              0
zip                  0
facility_count       0
county               0
fips_x               0
state_y           5349
fips_y            5349
dtype: int64

## Aggregate to county level

In [13]:
facility_county = (
    facilities
    .groupby("zip")
    ["facility_count"]
    .sum()
    .reset_index()
)

In [14]:
facilities['fips'] = facility_county['zip'].astype(int)

In [15]:
facility_county.head(2)

,zip,facility_count
0,10001,3
1,10002,5


In [16]:
cdc_24["fips"] = cdc_24["fips"].astype(str).str.replace('\.0', '', regex=True).str.zfill(5)

In [17]:
cdc_24["fips"].head()

0    01073
1    01073
2    01073
3    01073
4    01073
Name: fips, dtype: object

In [18]:
census_24["fips"].head()

0    01001
1    01003
2    01005
3    01007
4    01009
Name: fips, dtype: object

In [19]:
facilities["fips"] = facilities["fips"].astype(object)

In [20]:
facilities["fips"].head()

0    10001
1    10002
2    10003
3    10009
4    10010
Name: fips, dtype: object

In [21]:
master = (
    cdc_24
    .merge(
        census_24,
        on="fips",
        how="inner"
    )
)

master.shape

(1452, 18)

In [22]:
master.head(2)

,county_x,fips,Sex,Sex Code,year,Year Code,Cause of death,Cause of death Code,Place of Death,Place of Death Code,deaths,county_name,median_income,population,state,county_y,poverty_rate,unemployment_rate
0,"Jefferson County, AL",01073,Female,F,2024.0,2024.0,Accidental poisoning by and exposure to narcot...,X42,Decedent's home,4.0,15.0,"Jefferson County, Alabama",66388,667755,1,73,15.7,4.7
1,"Jefferson County, AL",01073,Female,F,2024.0,2024.0,Accidental poisoning by and exposure to narcot...,X42,Other,7.0,12.0,"Jefferson County, Alabama",66388,667755,1,73,15.7,4.7


### Now that I have my master data frame I can add treatment access

In [23]:
master = (
    master
    .merge(
        facilities,
        on="fips",
        how="left"
    )
)

#fill counties with no facilities
master["facility_count"] = (
    master["facility_count"]
    .fillna(0)
)

In [25]:
master.shape

(1452, 25)

In [26]:
master.head()

,county_x,fips,Sex,Sex Code,year,Year Code,Cause of death,Cause of death Code,Place of Death,Place of Death Code,...,county_y,poverty_rate,unemployment_rate,state_x,zip,facility_count,county,fips_x,state_y,fips_y
0,"Jefferson County, AL",01073,Female,F,2024.0,2024.0,Accidental poisoning by and exposure to narcot...,X42,Decedent's home,4.0,...,73,15.7,4.7,NaN,NaN,0.0,NaN,NaN,NaN,NaN
1,"Jefferson County, AL",01073,Female,F,2024.0,2024.0,Accidental poisoning by and exposure to narcot...,X42,Other,7.0,...,73,15.7,4.7,NaN,NaN,0.0,NaN,NaN,NaN,NaN
2,"Jefferson County, AL",01073,Female,F,2024.0,2024.0,Accidental poisoning by and exposure to other ...,X44,Decedent's home,4.0,...,73,15.7,4.7,NaN,NaN,0.0,NaN,NaN,NaN,NaN
3,"Jefferson County, AL",01073,Female,F,2024.0,2024.0,Accidental poisoning by and exposure to other ...,X44,Other,7.0,...,73,15.7,4.7,NaN,NaN,0.0,NaN,NaN,NaN,NaN
4,"Jefferson County, AL",01073,Male,M,2024.0,2024.0,Accidental poisoning by and exposure to narcot...,X42,Medical Facility - Outpatient or ER,2.0,...,73,15.7,4.7,NaN,NaN,0.0,NaN,NaN,NaN,NaN


In [27]:
#analysis variable
master["overdose_rate"] = (
    master["deaths"]
    /
    master["population"]
) * 100000

In [28]:
master["facility_rate"] = (
    master["facility_count"]
    /
    master["population"]
) * 100000